### Configure PyTorch CUDA memory allocation settings to prevent fragmentation

In [ ]:
 import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:128"

### Check GPU availability and display GPU information

In [1]:
gpu_info = !nvidia-smi
gpu_info = "\n".join(gpu_info)
if gpu_info.find("failed") >= 0:
    print("Not connected to a GPU")
else:
    print(gpu_info)

Sun Apr  5 14:37:03 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   29C    P0             44W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

### Install required Python packages (datasets, transformers, torchaudio, jiwer, accelerate, evaluate)

In [2]:
 %%capture
!pip install datasets==3.6.0
!pip install transformers==4.57.1
!pip install torchaudio
!pip install jiwer
!pip install accelerate -U
!pip install evaluate

### Install system dependencies and clone the KenLM language model toolkit

In [3]:
!sudo apt-get update
!sudo apt-get install build-essential libboost-all-dev cmake zlib1g-dev libbz2-dev liblzma-dev
!sudo apt-get install libboost-all-dev libeigen3-dev
!git clone https://github.com/kpu/kenlm
%cd kenlm
!pip install e .

Hit:1 https://cli.github.com/packages stable InRelease
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,980 kB]
Get:7 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:10 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,482 kB]
Hit:11 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,960 kB]
Hit:13 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jamm

### Build and install KenLM from source

In [4]:
!mkdir build
%cd build
!cmake ..
!make -j 4
!sudo make install

mkdir: cannot create directory ‘build’: File exists
/content/kenlm/build
-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
CMake Warning (dev) at CMakeLists.txt:101 (find_package):
  Policy CMP0167 is not set: The FindBoost module is removed.  Run "cmake
  --help-policy CMP0167" for policy details.  Use the cmake_policy command to
  set the policy and suppress this warning.

This warning is for project developers.  Use -Wno-dev to suppress it.

-- Found Boost: /usr/lib/x86_64-linux-gnu/cmake/Boost-1.74.0/BoostConfig.cmake (found suitable v

### Add KenLM binaries to the system PATH and verify installation

In [5]:
import os
os.environ['PATH'] += ":/content/kenlm/build/bin"  # Use your exact path
!echo $PATH  # Verify
!which lmplz  # Test

/opt/bin:/usr/local/cuda/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/tools/node/bin:/tools/google-cloud-sdk/bin:/content/kenlm/build/bin
/usr/local/bin/lmplz


### Initialize the CTC tokenizer, SeamlessM4T feature extractor, and Wav2Vec2-BERT processor

In [6]:
import re
import json
import unicodedata
import numpy as np
import torch
from dataclasses import dataclass
from typing import Dict, List, Union

from datasets import load_dataset, Audio, concatenate_datasets
import evaluate

from transformers import (
    Wav2Vec2CTCTokenizer,
    SeamlessM4TFeatureExtractor,
    Wav2Vec2BertProcessor,
    Wav2Vec2BertForCTC,
    TrainingArguments,
    Trainer,
)

### Load the Dhivehi Common Voice dataset and remove unused metadata columns

In [7]:
dv_train = load_dataset("fsicoli/common_voice_22_0", "dv", split="train+validation", trust_remote_code=True)
dv_test  = load_dataset("fsicoli/common_voice_22_0", "dv", split="test", trust_remote_code=True)

dv_train = dv_train.remove_columns(["accent","age","client_id","down_votes","gender","locale","segment","up_votes"])
dv_test  = dv_test.remove_columns(["accent","age","client_id","down_votes","gender","locale","segment","up_votes"])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

common_voice_22_0.py: 0.00B [00:00, ?B/s]

languages.py: 0.00B [00:00, ?B/s]

release_stats.py: 0.00B [00:00, ?B/s]

...


Reading metadata...: 2222it [00:00, 118840.93it/s]


Generating other split: 0 examples [00:00, ? examples/s]


Reading metadata...: 0it [00:00, ?it/s]
Reading metadata...: 15104it [00:00, 128509.93it/s]


Generating invalidated split: 0 examples [00:00, ? examples/s]


Reading metadata...: 1652it [00:00, 108734.39it/s]


### Install the wget package for downloading files

In [8]:
!pip install wget

  Preparing metadata (setup.py) ... done
  Created wheel for wget: filename=wget-3.2-py3-none-any.whl size=9655 sha256=10c66a459dced0fee27542c54b556c1056067a52d049b7b71890620ccead4a3a
  Stored in directory: /root/.cache/pip/wheels/01/46/3b/e29ffbe4ebe614ff224bad40fc6a5773a67a163251585a13a9
Successfully built wget


### Load the Sinhala ASR dataset with specified train/test splits

In [38]:
import os
import aiohttp
from datasets import load_dataset

os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "180"
os.environ["HF_HUB_ETAG_TIMEOUT"] = "180"

splits = load_dataset(
    "keshan/large-sinhala-asr-dataset",
    "si",
    split={
        "train": "train[:20%]",
        "test": "test[:20%]",
    },
    trust_remote_code=True,
    storage_options={
        "client_kwargs": {
            "timeout": aiohttp.ClientTimeout(total=3600)
        }
    },
)

si_train = splits["train"].rename_column("file", "audio").remove_columns(["filename", "x"])
si_test  = splits["test"].rename_column("file", "audio").remove_columns(["filename", "x"])


print("Sinhala train size:", len(si_train))
print("Sinhala test size:", len(si_test))

Sinhala train size: 26509
Sinhala test size: 4678


### Define and apply text cleaning: remove punctuation, special characters, and filter out Arabic script rows

In [10]:
 # -------- Dhivehi cleaning --------
chars_to_remove_regex_dv = r'[\,\?\.\!\-\;\:\"\“\%\‘\”\�\'\»\«]'
arabic_pattern = re.compile(r"[\u0600-\u06FF]")

def dv_clean(batch):
    s = re.sub(chars_to_remove_regex_dv, "", batch["sentence"]).lower()
    # remove Arabic rows
    if arabic_pattern.search(s):
        batch["sentence"] = ""
        return batch
    # remove latin
    s = re.sub(r"[a-z]+", "", s)
    batch["sentence"] = s.strip()
    return batch

# -------- Sinhala cleaning --------
chars_to_remove_regex_si = r'[\,\?\.\!\-\;\:\"\“\%\‘\”\�\(\)\'\”\‘‘\’\ʻ\”\“\–\—\…\»\«।॥]'
sinhala_allowed = re.compile(r"^[\u0D80-\u0DFF\s]+$")

def si_clean(batch):
    s = unicodedata.normalize("NFC", batch["sentence"])
    s = re.sub(chars_to_remove_regex_si, "", s)
    s = re.sub(r"[A-Za-z0-9]", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    batch["sentence"] = s
    return batch

### Apply cleaning functions and filter out empty sentences from both Dhivehi and Sinhala datasets

In [11]:
dv_train = dv_train.map(dv_clean).filter(lambda x: len(x["sentence"]) > 0)
dv_test  = dv_test.map(dv_clean).filter(lambda x: len(x["sentence"]) > 0)

si_train = si_train.map(si_clean).filter(lambda x: bool(sinhala_allowed.match(x["sentence"])) if x["sentence"] else False)
si_test  = si_test.map(si_clean).filter(lambda x: bool(sinhala_allowed.match(x["sentence"])) if x["sentence"] else False)

Map:   0%|          | 0/4897 [00:00<?, ? examples/s]

Filter:   0%|          | 0/4897 [00:00<?, ? examples/s]

Map:   0%|          | 0/2222 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2222 [00:00<?, ? examples/s]

Map:   0%|          | 0/26509 [00:00<?, ? examples/s]

Filter:   0%|          | 0/26509 [00:00<?, ? examples/s]

Map:   0%|          | 0/4678 [00:00<?, ? examples/s]

Filter:   0%|          | 0/4678 [00:00<?, ? examples/s]

### Resample all audio to 16kHz for both Dhivehi and Sinhala datasets

In [12]:
dv_train = dv_train.cast_column("audio", Audio(sampling_rate=16_000))
dv_test  = dv_test.cast_column("audio", Audio(sampling_rate=16_000))
si_train = si_train.cast_column("audio", Audio(sampling_rate=16_000))
si_test  = si_test.cast_column("audio", Audio(sampling_rate=16_000))

### Extract the combined character vocabulary from both Dhivehi and Sinhala datasets

In [13]:
def extract_all_chars(batch):
    all_text = " ".join(batch["sentence"])
    vocab = list(set(all_text))
    return {"vocab": [vocab], "all_text": [all_text]}

all_train_text = concatenate_datasets([
    dv_train.select_columns(["sentence"]),
    si_train.select_columns(["sentence"]),
])
all_test_text = concatenate_datasets([
    dv_test.select_columns(["sentence"]),
    si_test.select_columns(["sentence"]),
])

vocab_train = all_train_text.map(
    extract_all_chars, batched=True, batch_size=-1, keep_in_memory=True,
    remove_columns=all_train_text.column_names
)
vocab_test = all_test_text.map(
    extract_all_chars, batched=True, batch_size=-1, keep_in_memory=True,
    remove_columns=all_test_text.column_names
)

vocab_list = list(set(vocab_train["vocab"][0]) | set(vocab_test["vocab"][0]))
vocab_dict = {v: k for k, v in enumerate(sorted(vocab_list))}

# Space -> word delimiter |
vocab_dict["|"] = vocab_dict[" "]
del vocab_dict[" "]


# Specials
vocab_dict["[UNK]"] = len(vocab_dict)
vocab_dict["[PAD]"] = len(vocab_dict)

with open("vocab.json", "w", encoding="utf-8") as vocab_file:
    json.dump(vocab_dict, vocab_file, ensure_ascii=False)

print("Vocab size:", len(vocab_dict))

Map:   0%|          | 0/26067 [00:00<?, ? examples/s]

Map:   0%|          | 0/5938 [00:00<?, ? examples/s]

Vocab size: 130


### Initialize the CTC tokenizer, SeamlessM4T feature extractor, and Wav2Vec2-BERT processor

In [14]:
tokenizer = Wav2Vec2CTCTokenizer.from_pretrained(
    "./",
    unk_token="[UNK]",
    pad_token="[PAD]",
    word_delimiter_token="|",
)

feature_extractor = SeamlessM4TFeatureExtractor(
    feature_size=80,
    num_mel_bins=80,
    sampling_rate=16000,
    padding_value=0.0,
)

processor = Wav2Vec2BertProcessor(feature_extractor=feature_extractor, tokenizer=tokenizer)

### Write all transcriptions to a text file for KenLM language model training

In [15]:
with open("lm_corpus.txt", "w", encoding="utf-8") as f:
    for ex in all_train_text:
        s = ex["sentence"].strip()
        if s:
            f.write(s + "\n")
    for ex in all_test_text:
        s = ex["sentence"].strip()
        if s:
            f.write(s + "\n")

print("Wrote LM corpus to lm_corpus.txt")

Wrote LM corpus to lm_corpus.txt


### Define the dataset preparation function: extract audio features and encode text labels, then apply to train and test sets

In [16]:
def prepare_dataset(batch):
    audio = batch["audio"]
    batch["input_features"] = processor(audio["array"], sampling_rate=audio["sampling_rate"]).input_features[0]
    batch["input_length"] = len(batch["input_features"])
    batch["labels"] = processor(text=batch["sentence"]).input_ids
    return batch

dv_train_p = dv_train.map(prepare_dataset, remove_columns=dv_train.column_names)
dv_test_p  = dv_test.map(prepare_dataset,  remove_columns=dv_test.column_names)
si_train_p = si_train.map(prepare_dataset, remove_columns=si_train.column_names)
si_test_p  = si_test.map(prepare_dataset,  remove_columns=si_test.column_names)

Map:   0%|          | 0/4639 [00:00<?, ? examples/s]

Map:   0%|          | 0/2107 [00:00<?, ? examples/s]

Map:   0%|          | 0/21428 [00:00<?, ? examples/s]

Map:   0%|          | 0/3831 [00:00<?, ? examples/s]

### Concatenate Dhivehi and Sinhala processed datasets into unified train and test sets

In [17]:
train = concatenate_datasets([dv_train_p, si_train_p]).shuffle(seed=42)
test  = concatenate_datasets([dv_test_p,  si_test_p]).shuffle(seed=42)

print("Train size:", len(train))
print("Test size:", len(test))


Train size: 26067
Test size: 5938


### Define a custom data collator that pads input features and labels for CTC training

In [18]:
@dataclass
class DataCollatorCTCWithPadding:
    processor: Wav2Vec2BertProcessor
    padding: Union[bool, str] = True

    def __call__(self, features: List[Dict]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": f["input_features"]} for f in features]
        label_features = [{"input_ids": f["labels"]} for f in features]

        batch = self.processor.pad(input_features, padding=self.padding, return_tensors="pt")
        labels_batch = self.processor.pad(labels=label_features, padding=self.padding, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(labels_batch["attention_mask"].ne(1), -100)
        batch["labels"] = labels
        return batch

data_collator = DataCollatorCTCWithPadding(processor=processor, padding=True)

### Train a trigram KenLM language model from the corpus

In [19]:
!lmplz -o 3 < lm_corpus.txt > lm.arpa

=== 1/5 Counting and sorting n-grams ===
Reading /content/kenlm/build/lm_corpus.txt
----5---10---15---20---25---30---35---40---45---50---55---60---65---70---75---80---85---90---95--100
****************************************************************************************************
Unigram tokens 153082 types 41491
=== 2/5 Calculating and sorting adjusted counts ===
Chain sizes: 1:497892 2:24940126208 3:46762737664
Statistics:
1 41491 D1=0.724133 D2=1.11098 D3+=1.32224
2 120888 D1=0.890628 D2=1.15597 D3+=1.39493
3 134489 D1=0.847419 D2=1.64897 D3+=2.16776
Memory estimate for binary LM:
type      kB
probing 6250 assuming -p 1.5
probing 7121 assuming -r models -p 1.5
trie    3175 without quantization
trie    2107 assuming -q 8 -b 8 quantization 
trie    3014 assuming -a 22 array pointer compression
trie    1946 assuming -a 22 -q 8 -b 8 array pointer compression and quantization
=== 3/5 Calculating and sorting initial probabilities ===
Chain sizes: 1:497892 2:1934208 3:2689780
----5---

### Install pyctcdecode library for CTC beam search decoding with language model support

In [20]:
!pip install pyctcdecode

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.5/529.5 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 130.3 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
tobler 0.13.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
pytensor 2.38.2 requires numpy

### Build a CTC beam search decoder with KenLM language model integration

In [21]:
from pyctcdecode import build_ctcdecoder

# get labels in ID order
vocab = processor.tokenizer.get_vocab()  # token -> id
id2token = {v: k for k, v in vocab.items()}
labels = [id2token[i] for i in range(len(id2token))]

# path to your trained KenLM model (arpa or binary)
kenlm_model_path = "lm.arpa"  # <-- change if you use another path/name

decoder = build_ctcdecoder(
    labels=labels,
    kenlm_model_path=kenlm_model_path,
    alpha=0.5,  # tune on dev
    beta=1.0  # tune on dev
)


### Define the evaluation metrics function using LM-aided beam search decoding to compute WER and CER

In [22]:
import evaluate
import torch

wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

def compute_metrics(pred):
    pred_logits = pred.predictions
    label_ids = pred.label_ids

    # Convert logits to probabilities
    probs = torch.softmax(torch.tensor(pred_logits), dim=-1).numpy()

    # Decode with LM
    pred_str = [decoder.decode(p, beam_width=64) for p in probs]

    # Replace -100 with pad_token_id
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    label_str = processor.batch_decode(label_ids, group_tokens=False)

    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    cer = cer_metric.compute(predictions=pred_str, references=label_str)

    return {"wer": wer, "cer": cer}

### Load the pre-trained W2V-BERT 2.0 model with an adapter and CTC head for fine-tuning

In [23]:
model = Wav2Vec2BertForCTC.from_pretrained(
    "facebook/w2v-bert-2.0",
    add_adapter=True,
    pad_token_id=processor.tokenizer.pad_token_id,
    vocab_size=len(processor.tokenizer),
)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/2.32G [00:00<?, ?B/s]

Some weights of Wav2Vec2BertForCTC were not initialized from the model checkpoint at facebook/w2v-bert-2.0 and are newly initialized: ['lm_head.bias', 'lm_head.weight', 'wav2vec2_bert.adapter.layers.0.ffn.intermediate_dense.bias', 'wav2vec2_bert.adapter.layers.0.ffn.intermediate_dense.weight', 'wav2vec2_bert.adapter.layers.0.ffn.output_dense.bias', 'wav2vec2_bert.adapter.layers.0.ffn.output_dense.weight', 'wav2vec2_bert.adapter.layers.0.ffn_layer_norm.bias', 'wav2vec2_bert.adapter.layers.0.ffn_layer_norm.weight', 'wav2vec2_bert.adapter.layers.0.residual_conv.bias', 'wav2vec2_bert.adapter.layers.0.residual_conv.weight', 'wav2vec2_bert.adapter.layers.0.residual_layer_norm.bias', 'wav2vec2_bert.adapter.layers.0.residual_layer_norm.weight', 'wav2vec2_bert.adapter.layers.0.self_attn.linear_k.bias', 'wav2vec2_bert.adapter.layers.0.self_attn.linear_k.weight', 'wav2vec2_bert.adapter.layers.0.self_attn.linear_out.bias', 'wav2vec2_bert.adapter.layers.0.self_attn.linear_out.weight', 'wav2vec2_ber

### Configure training hyperparameters: batch size, learning rate, evaluation strategy, checkpointing, and early stopping

In [24]:
from transformers import TrainingArguments, Trainer, Wav2Vec2BertForCTC, EarlyStoppingCallback

training_args = TrainingArguments(
    output_dir="./outputs",
    group_by_length=True,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=2,
    num_train_epochs=10,
    gradient_checkpointing=True,
    fp16=True,

    eval_strategy="steps",     # FIXED
    eval_steps=300,

    logging_strategy="steps",
    logging_steps=300,

    save_strategy="steps",
    save_steps=2700,

    learning_rate=5e-5,
    warmup_steps=500,
    save_total_limit=2,

    load_best_model_at_end=True,
    metric_for_best_model="eval_wer",   # FIXED
    greater_is_better=False,

    push_to_hub=False,
    report_to="none",
)

trainer = Trainer(
    model=model,
    data_collator=data_collator,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=train,
    eval_dataset=test,
    tokenizer=processor
)


/tmp/ipykernel_664/1422347001.py:33: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


### Start the fine-tuning training run

In [25]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 131, 'bos_token_id': 130}.


Step,Training Loss,Validation Loss,Wer,Cer
300,2777.859600,319.780518,0.941549,0.242370
600,943.875400,172.952133,0.635881,0.134273
900,632.305500,134.620605,0.517595,0.106480
1200,459.468500,120.860008,0.494728,0.100301
1500,458.165300,108.829048,0.466227,0.092561
1800,384.141400,105.954628,0.454168,0.089451
2100,360.801700,100.721939,0.438353,0.085133
2400,327.431700,96.028206,0.420857,0.079138


Step,Training Loss,Validation Loss,Wer,Cer
300,2777.859600,319.780518,0.941549,0.242370
600,943.875400,172.952133,0.635881,0.134273
900,632.305500,134.620605,0.517595,0.106480
1200,459.468500,120.860008,0.494728,0.100301
1500,458.165300,108.829048,0.466227,0.092561
1800,384.141400,105.954628,0.454168,0.089451
2100,360.801700,100.721939,0.438353,0.085133
2400,327.431700,96.028206,0.420857,0.079138
2700,306.209900,94.448784,0.413015,0.076593
3000,247.719200,89.578117,0.399110,0.074728


TrainOutput(global_step=8150, training_loss=417.23020417944787, metrics={'train_runtime': 17729.7613, 'train_samples_per_second': 14.702, 'train_steps_per_second': 0.46, 'total_flos': 3.4309094886272336e+19, 'train_loss': 417.23020417944787, 'epoch': 10.0})

### Mount Google Drive for persistent storage

In [27]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Copy the best checkpoint to Google Drive for persistent storage

In [28]:
!cp -r ./outputs/checkpoint-8150 /content/drive/MyDrive/final_model90

### Run greedy CTC inference on the test set and compute WER/CER

In [30]:
from transformers import AutoModelForCTC, Wav2Vec2BertProcessor
import torch
import numpy as np

all_pred = []
all_refs = []

checkpoint_dir = "/content/kenlm/build/outputs/checkpoint-8150"

model = AutoModelForCTC.from_pretrained(checkpoint_dir).to("cuda")
processor = Wav2Vec2BertProcessor.from_pretrained(checkpoint_dir)

model.eval()

for ex in dv_test_p:
    input_feats = torch.tensor(ex["input_features"]).unsqueeze(0).to(model.device)

    with torch.no_grad():
        logits = model(input_feats).logits

    # Convert logits to probabilities for LM decoding
    probs = torch.softmax(logits[0], dim=-1).cpu().numpy()

    # Decode with language model
    pred_text = decoder.decode(probs)

    # Reference text
    ref_text = processor.decode(ex["labels"], group_tokens=False)

    all_pred.append(pred_text)
    all_refs.append(ref_text)

wer = wer_metric.compute(predictions=all_pred, references=all_refs)
cer = cer_metric.compute(predictions=all_pred, references=all_refs)

print(f"WER (Dhivehi only, LM decode): {wer:.4f}")
print(f"CER (Dhivehi only, LM decode): {cer:.4f}")

for idx in range(min(3, len(all_refs))):
    print("\n--- Example", idx, "---")
    print("REF :", all_refs[idx])
    print("PRED:", all_pred[idx])

WER (Dhivehi only, LM decode): 0.1334
CER (Dhivehi only, LM decode): 0.0304

--- Example 0 ---
REF : ޕިކަޕް މަރާމާތުކޮށްދޭނެ ފަރާތެއް ހޯދުން
PRED: ޕިކަޕް މަރާމާތުކޮށްދޭނެ ފަރާތެއް ހޯދުން

--- Example 1 ---
REF : އަންޖޫދާ މަޑުމަޑުން އެހެން ބުނެލުމުން ރިޝްކާ ވެސް އިތުރު ސުވާލެއް ނުކުރޭ
PRED: އަންޖޫދާ މަޑުމަޑުން އެހެން ބުނެލުމުން ރިޝްކާ ވެސް އިތުރު ސުވާލެއް ނުކުރޭ

--- Example 2 ---
REF : އެސްފިޔަތަކުން ވަނީ ނިދި ގެއްލިފަ
PRED: އެސްފިޔަތަކުން ވަނީ ނިދި ގެއްލިފަ


### Run greedy CTC inference on the test set and compute WER/CER

In [31]:
from transformers import AutoModelForCTC, Wav2Vec2BertProcessor
import torch
import numpy as np

all_pred = []
all_refs = []

checkpoint_dir = "/content/kenlm/build/outputs/checkpoint-8150"

model = AutoModelForCTC.from_pretrained(checkpoint_dir).to("cuda")
processor = Wav2Vec2BertProcessor.from_pretrained(checkpoint_dir)

model.eval()

for ex in si_test_p:
    input_feats = torch.tensor(ex["input_features"]).unsqueeze(0).to(model.device)

    with torch.no_grad():
        logits = model(input_feats).logits

    # Convert logits to probabilities for LM decoding
    probs = torch.softmax(logits[0], dim=-1).cpu().numpy()

    # Decode with language model
    pred_text = decoder.decode(probs)

    # Reference text
    ref_text = processor.decode(ex["labels"], group_tokens=False)

    all_pred.append(pred_text)
    all_refs.append(ref_text)

wer = wer_metric.compute(predictions=all_pred, references=all_refs)
cer = cer_metric.compute(predictions=all_pred, references=all_refs)

print(f"WER (Sinhala only, LM decode): {wer:.4f}")
print(f"CER (Sinhala only, LM decode): {cer:.4f}")

for idx in range(min(3, len(all_refs))):
    print("\n--- Example", idx, "---")
    print("REF :", all_refs[idx])
    print("PRED:", all_pred[idx])

WER (Sinhala only, LM decode): 0.0835
CER (Sinhala only, LM decode): 0.0226

--- Example 0 ---
REF : ආචාර්යවරිය නම් ලකුණු කරන අතරතුර
PRED: ආචාර්යවරිය නම් ලකුණු කරන අතරතුර

--- Example 1 ---
REF : මේ අයුරින් මරණයට පත් වේ
PRED: මේ අයුරින් මරණයට පත් වේ

--- Example 2 ---
REF : ඒකට මොකද මේකෙන්
PRED: ඒකට මොකද මේකෙන්
